# quality-voice-colab: F5-TTS + your Applio voice, on a free T4

**Experiment, not part of cyv v1.** This is the "quality tier" from
`docs/adr/0005-layered-voice-pipeline.md`, run on Colab instead of your 8 GB Mac.

For every reference clip and every line in your list, this notebook:

1. makes speech with **F5-TTS** (it copies tone from the reference clip), then
2. runs that speech through **Applio** with your trained voice model (`sachinkoli2`).

It writes the results to Google Drive, plus a shuffled **blind listening set**.

**Before you run it**

- Put your files in Google Drive first (see `README.md` in this folder): `refs/` and `lines.txt`.
- Runtime > Change runtime type > **T4 GPU**.
- **F5-TTS weights are probably non-commercial (CC-BY-NC).** Use this for personal listening only.
- Only your own voice, or a voice whose owner gave you permission. See `USAGE_POLICY.md`.
- **Before saving this notebook back to GitHub:** Edit > **Clear all outputs**. Otherwise your audio
  and transcripts end up in the repository.

## 1. Check that a GPU is attached

F5-TTS and Applio both need a GPU. Without one, each line would take far too long.
If this cell fails, change the runtime to a T4 GPU and run it again.

In [ ]:
# @title Check the GPU
import shutil
import subprocess


def gpu_name():
    if not shutil.which("nvidia-smi"):
        return None
    r = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True,
        check=False,
    )
    return r.stdout.strip() if r.returncode == 0 and r.stdout.strip() else None


GPU = gpu_name()
if GPU is None:
    raise RuntimeError(
        "No GPU found.\n"
        "  1. Runtime > Change runtime type > select 'T4 GPU' > Save, then run this cell again.\n"
        "  2. If T4 is greyed out or no GPU is available, your free GPU quota is used up. It resets\n"
        "     after a while. Come back later, or use Colab Pro."
    )
print("GPU OK:", GPU)

## 2. Settings and Google Drive

Change the values in the first block if your Drive folders are named differently.
A popup will ask you to allow Drive access. If it fails, run this cell again.

- `DRIVE_DIR`: the project folder in your Drive.
- `VOICE`: the voice model folder inside it (same name as in `train_colab.ipynb`).
- `STEPS`: F5 quality. 32 is what sounded right in our tests. Lower is faster and worse.
- `SEED`: change it for a different take of the same lines.
- `PITCH`: Applio pitch shift. 0 keeps your model's own pitch.

In [ ]:
# @title Settings and Google Drive
import re
from pathlib import Path

DRIVE_DIR = "/content/drive/MyDrive/clone-your-voice"  # @param {type:"string"}
VOICE = "sachinkoli2"  # @param {type:"string"}
STEPS = 32  # @param {type:"integer"}
SEED = 42  # @param {type:"integer"}
PITCH = 0  # @param {type:"integer"}

if not re.fullmatch(r"[a-z0-9_-]{1,32}", VOICE):
    raise ValueError("VOICE must be 1 to 32 characters: lowercase letters, digits, _ or -.")
if not 4 <= STEPS <= 64:
    raise ValueError("STEPS must be between 4 and 64. 32 is the tested value.")

REF_DIR = f"{DRIVE_DIR}/refs"  # <name>.wav + <name>.txt pairs
LINES_FILE = f"{DRIVE_DIR}/lines.txt"  # one sentence per line
OUT_DIR = f"{DRIVE_DIR}/quality-out"  # written by this notebook
VOICE_DIR = f"{DRIVE_DIR}/{VOICE}"  # the trained Applio model folder

try:
    from google.colab import drive

    drive.mount("/content/drive")
except Exception as exc:  # noqa: BLE001
    raise RuntimeError(
        "Could not mount Google Drive (" + repr(exc) + ").\n"
        "Run this cell again and approve the popup."
    ) from exc

if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Drive mounted but /content/drive/MyDrive is missing. Run the cell again.")
print("Drive OK. Project folder:", DRIVE_DIR)

## 3. Check your inputs

This runs before the long installs, so a missing file stops the notebook early.
It checks:

- every `refs/<name>.wav` has a `<name>.txt` with the exact words spoken in it;
- `lines.txt` has at least one sentence;
- the voice model exists. `train_colab.ipynb` saves it in `<VOICE>/checkpoints/` as
  `<VOICE>_<epoch>e_<step>s.pth` and `<VOICE>.index`. The notebook also looks in `<VOICE>/` itself.

It prints file names and counts only, never the text.

In [ ]:
# @title Check refs, lines and the voice model
import re
import wave
from pathlib import Path

REFS = []
for wav in sorted(Path(REF_DIR).glob("*.wav")):
    if not re.fullmatch(r"[A-Za-z0-9_-]{1,40}", wav.stem):
        raise RuntimeError(f"Rename {wav.name}: use only letters, digits, _ and - in the name.")
    txt = wav.with_suffix(".txt")
    if not txt.is_file():
        raise RuntimeError(f"{wav.name} has no {txt.name} next to it (the exact transcript).")
    text = txt.read_text(encoding="utf-8").strip()
    if not text:
        raise RuntimeError(f"{txt.name} is empty. It must hold the exact words in {wav.name}.")
    try:
        with wave.open(str(wav), "rb") as w:
            seconds = w.getnframes() / w.getframerate()
        if not 8 <= seconds <= 12:
            print(f"WARNING: {wav.name} is {seconds:.1f} s. 8 to 12 s works best.")
    except wave.Error:
        print(f"NOTE: could not read the length of {wav.name} (not plain WAV). Check it by ear.")
    REFS.append({"name": wav.stem, "wav": str(wav), "text": text})
if not REFS:
    raise RuntimeError(f"No reference clips in {REF_DIR}. Add <name>.wav and <name>.txt pairs.")

lines_path = Path(LINES_FILE)
if not lines_path.is_file():
    raise RuntimeError(f"{LINES_FILE} not found. Make it with one sentence per line.")
LINES = [ln.strip() for ln in lines_path.read_text(encoding="utf-8").splitlines() if ln.strip()]
if not LINES:
    raise RuntimeError(f"{LINES_FILE} has no sentences.")
LINE_JOBS = [(i, text) for i, text in enumerate(LINES, start=1)]


def find_model(voice_dir, name):
    """Highest-epoch small model, the same rule as train_colab.ipynb (never the G_/D_ files)."""
    rx = re.compile(rf"^{re.escape(name)}_(\d+)e_(\d+)s\.pth$")
    best, best_key = None, (-1, -1)
    for folder in (voice_dir / "checkpoints", voice_dir):
        if not folder.is_dir():
            continue
        for p in folder.glob("*.pth"):
            m = rx.match(p.name)
            if m and (int(m[1]), int(m[2])) > best_key:
                best, best_key = p, (int(m[1]), int(m[2]))
    return best


def find_index(voice_dir, name):
    for folder in (voice_dir / "checkpoints", voice_dir):
        p = folder / f"{name}.index"
        if p.is_file():
            return p
    return None


MODEL_PTH = find_model(Path(VOICE_DIR), VOICE)
MODEL_INDEX = find_index(Path(VOICE_DIR), VOICE)
if MODEL_PTH is None or MODEL_INDEX is None:
    raise RuntimeError(
        f"Could not find the model for '{VOICE}' under {VOICE_DIR}.\n"
        f"Expected {VOICE}_<epoch>e_<step>s.pth and {VOICE}.index in {VOICE_DIR}/checkpoints/ "
        f"(where train_colab.ipynb saves them) or in {VOICE_DIR}/."
    )

print(len(REFS), "reference clip(s):", ", ".join(r["name"] for r in REFS))
print(len(LINES), "line(s) to speak")
print("Model:", MODEL_PTH.name, "| index:", MODEL_INDEX.name)

## 4. Install F5-TTS

F5-TTS gets its **own** virtual environment (`/content/venv_f5`), so it cannot break Applio's.
We pin `f5-tts==1.1.22`. We do not use `constraints.txt` here: its `pillow==13.0.0` clashes with
the `gradio` that F5-TTS installs, so the install fails if we try.

Takes a few minutes. The first synthesis run downloads the F5 weights.

In [ ]:
# @title Install F5-TTS (own virtual environment)
import subprocess  # noqa: F811 (each cell can run on its own)
import sys
from pathlib import Path

F5_VENV = Path("/content/venv_f5")
F5_PY = str(F5_VENV / "bin" / "python")
F5_VERSION = "f5-tts==1.1.22"


def sh(cmd):
    print("$", " ".join(str(c) for c in cmd))
    subprocess.run([str(c) for c in cmd], check=True)


marker = F5_VENV / ".ready"
if marker.is_file() and marker.read_text().strip() == F5_VERSION:
    print("F5-TTS already installed, skipping.")
else:
    sh([sys.executable, "-m", "pip", "install", "-q", "uv"])
    sh(["uv", "venv", F5_VENV, "--python", "3.11", "--clear"])
    sh(["uv", "pip", "install", "-q", "--python", F5_PY, F5_VERSION])
    marker.write_text(F5_VERSION)

sh(
    [
        F5_PY,
        "-c",
        "import torch, f5_tts.api; print('F5 OK, torch', torch.__version__, 'CUDA:', torch.cuda.is_available())",
    ]
)

## 5. Install Applio at the pinned commit

This is the same install as `train_colab.ipynb`:

- Applio is fetched at **one exact commit**, not "latest". It is the same commit as in the repo's
  Applio pin file and in `train_colab.ipynb`. If you change one, change all of them.
- Its own virtual environment is `/content/applio-venv`, with `constraints.txt` from the repo
  (checked against a SHA-256 hash first).
- Applio's `prerequisites` command downloads the models it needs (RMVPE pitch model, ContentVec).

Takes 5 to 10 minutes. After a runtime restart, run it again. It skips what is already there.

In [ ]:
# @title Install Applio at the pinned commit
import hashlib
import subprocess  # noqa: F811 (each cell can run on its own)
import sys
import urllib.request
from pathlib import Path

# Must equal APPLIO_COMMIT in the repo's Applio pin file and in notebooks/train_colab.ipynb.
APPLIO_REPO = "https://github.com/IAHispano/Applio"
APPLIO_COMMIT = "c817f1f83f689ef72e0dfeb1c17ae990b9248506"

CONSTRAINTS_URL = (
    "https://raw.githubusercontent.com/kolisachint/clone-your-voice/main/notebooks/constraints.txt"
)
CONSTRAINTS_SHA256 = "b6d5e81f1e02c787276aab11835eb5b5c9e69db3b9970c93d00c2bb6b3176e40"

APPLIO = Path("/content/Applio")
VENV = Path("/content/applio-venv")
PY = str(VENV / "bin" / "python")
CONSTRAINTS = Path("/content/constraints.txt")


def sh(cmd, cwd=None):
    print("$", " ".join(str(c) for c in cmd))
    subprocess.run([str(c) for c in cmd], cwd=cwd, check=True)


with urllib.request.urlopen(CONSTRAINTS_URL, timeout=60) as resp:  # noqa: S310
    blob = resp.read()
if hashlib.sha256(blob).hexdigest() != CONSTRAINTS_SHA256:
    raise RuntimeError(
        "constraints.txt does not match the expected hash. Re-open the latest notebook."
    )
CONSTRAINTS.write_bytes(blob)

marker = VENV / ".ready"
if marker.is_file() and marker.read_text().strip() == APPLIO_COMMIT:
    print("Applio already installed for this commit, skipping.")
else:
    APPLIO.mkdir(exist_ok=True)
    if not (APPLIO / ".git").is_dir():
        sh(["git", "init", "-q"], cwd=APPLIO)
        sh(["git", "remote", "add", "origin", APPLIO_REPO], cwd=APPLIO)
    sh(["git", "fetch", "--depth", "1", "origin", APPLIO_COMMIT], cwd=APPLIO)
    sh(["git", "-c", "advice.detachedHead=false", "checkout", "-q", "FETCH_HEAD"], cwd=APPLIO)
    head = subprocess.run(
        ["git", "rev-parse", "HEAD"], cwd=APPLIO, capture_output=True, text=True, check=True
    ).stdout.strip()
    if head != APPLIO_COMMIT:
        raise RuntimeError(f"Applio is at {head}, expected {APPLIO_COMMIT}")

    sh(["apt-get", "install", "-y", "-qq", "portaudio19-dev"])
    sh([sys.executable, "-m", "pip", "install", "-q", "uv"])
    sh(["uv", "venv", VENV, "--python", "3.12", "--seed", "--allow-existing"])
    sh(
        [
            "uv",
            "pip",
            "install",
            "-q",
            "-r",
            APPLIO / "requirements-engine.txt",
            "-c",
            CONSTRAINTS,
            "--extra-index-url",
            "https://download.pytorch.org/whl/cu128",
            "--index-strategy",
            "unsafe-best-match",
            "--python",
            PY,
        ]
    )
    sh(
        [PY, "core.py", "prerequisites", "--models", "--pretraineds-hifigan", "--no-exe"],
        cwd=APPLIO,
    )
    marker.write_text(APPLIO_COMMIT)

sh([PY, "core.py", "infer", "--help"], cwd=APPLIO)

## 6. Make the F5 clips (every reference x every line)

F5-TTS runs in its own environment. It loads once and writes
`quality-out/<reference>/<NN>_f5.wav` for each line. Its settings match `f5say.py`:
32 steps, cfg 2.0, sway -1.0, and the same length estimate from the reference's speaking rate.
(The PyTorch F5 package does that estimate itself when no fixed length is given.)

Finished files are skipped, so you can run it again after a disconnect.
The timing per line is printed after each clip.

In [ ]:
# @title F5-TTS: one clip per reference and line
import json
import subprocess
from pathlib import Path

WORK = Path("/content/quality")
WORK.mkdir(parents=True, exist_ok=True)
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

F5_SCRIPT = '''
"""Runs inside the F5 venv: one F5-TTS clip per (reference, line).

Prints file names and timings only, never the text.
"""
import json
import sys
import time
from pathlib import Path

from f5_tts.api import F5TTS

job = json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))
out_root = Path(job["out_dir"])


def quiet(*_args, **_kwargs):
    """F5 can echo the reference transcript in its messages. Drop them."""


tts = F5TTS()  # first run downloads the F5 weights
for ref in job["refs"]:
    ref_dir = out_root / ref["name"]
    ref_dir.mkdir(parents=True, exist_ok=True)
    for nn, line in job["lines"]:
        target = ref_dir / f"{nn:02d}_f5.wav"
        if target.exists():
            print("skip", ref["name"] + "/" + target.name, "(already there)", flush=True)
            continue
        t0 = time.time()
        tts.infer(
            ref_file=ref["wav"],
            ref_text=ref["text"],
            gen_text=line,
            show_info=quiet,
            file_wave=str(target),
            nfe_step=job["steps"],
            cfg_strength=2.0,
            sway_sampling_coef=-1.0,
            speed=1.0,
            target_rms=0.1,
            cross_fade_duration=0.15,
            remove_silence=False,
            seed=job["seed"],
        )
        print(ref["name"] + "/" + target.name, f"{time.time() - t0:.1f} s", flush=True)
'''
(WORK / "run_f5.py").write_text(F5_SCRIPT, encoding="utf-8")

job = {
    "out_dir": OUT_DIR,
    "steps": STEPS,
    "seed": SEED,
    "refs": REFS,
    "lines": [[nn, text] for nn, text in LINE_JOBS],
}
job_path = WORK / "job.json"
job_path.write_text(json.dumps(job), encoding="utf-8")  # text stays on this runtime's disk

subprocess.run([F5_PY, str(WORK / "run_f5.py"), str(job_path)], check=True)
made = sorted(Path(OUT_DIR).glob("*/*_f5.wav"))
print(len(made), "F5 clips in", OUT_DIR)

## 7. Convert each F5 clip with your voice (Applio)

Applio's **batch** command loads your model once per reference, then converts every clip in
that batch. The per-line time printed here is the batch time divided by the number of lines.
It is an average, not a per-clip measurement.

Each result is written to `quality-out/<reference>/<NN>_f5_rvc.wav`.

In [ ]:
# @title Applio: your voice on each F5 clip
import shutil  # noqa: F811 (each cell can run on its own)
import subprocess
import time
from pathlib import Path


def sh(cmd, cwd=None):
    print("$", " ".join(str(c) for c in cmd))
    subprocess.run([str(c) for c in cmd], cwd=cwd, check=True)


rvc_in_root = Path("/content/quality/rvc_in")
rvc_out_root = Path("/content/quality/rvc_out")
for ref in REFS:
    out_ref = Path(OUT_DIR) / ref["name"]
    todo = [
        p
        for p in sorted(out_ref.glob("*_f5.wav"))
        if not (out_ref / p.name.replace("_f5.wav", "_f5_rvc.wav")).is_file()
    ]
    if not todo:
        print(ref["name"], "is already converted, skipping.")
        continue

    in_dir = rvc_in_root / ref["name"]
    out_dir = rvc_out_root / ref["name"]
    shutil.rmtree(in_dir, ignore_errors=True)
    shutil.rmtree(out_dir, ignore_errors=True)
    in_dir.mkdir(parents=True)
    out_dir.mkdir(parents=True)
    for p in todo:
        shutil.copyfile(p, in_dir / p.name)

    t0 = time.time()
    sh(
        [
            PY,
            "core.py",
            "batch-infer",
            "--input-folder",
            in_dir,
            "--output-folder",
            out_dir,
            "--pth-path",
            MODEL_PTH,
            "--index-path",
            MODEL_INDEX,
            "--pitch",
            PITCH,
            "--f0-method",
            "rmvpe",
            "--embedder-model",
            "contentvec",
            "--export-format",
            "WAV",
        ],
        cwd=APPLIO,
    )
    elapsed = time.time() - t0

    for p in todo:
        produced = out_dir / (p.stem + "_output.wav")  # Applio names its output <stem>_output.wav
        if not produced.is_file():
            raise RuntimeError(
                f"Applio did not write {produced.name} for {ref['name']}. Scroll up for its error."
            )
        shutil.copyfile(produced, out_ref / p.name.replace("_f5.wav", "_f5_rvc.wav"))
    print(
        ref["name"],
        f"{len(todo)} line(s) in {elapsed:.0f} s,",
        f"average {elapsed / len(todo):.1f} s per line",
    )

## 8. Blind listening set

The notebook copies every clip (F5 and F5 + your voice, for every reference) into
`quality-out/blind/` under shuffled names, one group per line: `L01_A.wav`, `L01_B.wav`, and so on.
The answer key is `quality-out/blind/key.txt`.

**Listen first, rank, and only then open `key.txt`.** Each run makes a new shuffle.

In [ ]:
# @title Write the blind listening set
import json
import random
import shutil
import string
from pathlib import Path

out_root = Path(OUT_DIR)
blind = out_root / "blind"
shutil.rmtree(blind, ignore_errors=True)
blind.mkdir(parents=True)

# A plain shuffle for a listening test, not security. Not seeded on purpose: new shuffle each run.
rng = random.Random()  # noqa: S311
key = {}
for nn, _text in LINE_JOBS:
    options = []
    for ref in REFS:
        for suffix in ("_f5.wav", "_f5_rvc.wav"):
            src = out_root / ref["name"] / f"{nn:02d}{suffix}"
            if src.is_file():
                options.append(f"{ref['name']}/{src.name}")
    if len(options) > len(string.ascii_uppercase):
        raise RuntimeError("Too many clips for one line to label A to Z. Use fewer references.")
    rng.shuffle(options)
    for letter, rel in zip(string.ascii_uppercase, options, strict=False):
        name = f"L{nn:02d}_{letter}.wav"
        shutil.copyfile(out_root / rel, blind / name)
        key[name] = rel

(blind / "key.txt").write_text(json.dumps(key, indent=2) + "\n", encoding="utf-8")
print(len(key), "blind clips in", blind)
print("Listen to them, rank them, then open key.txt.")

## 9. Next steps

1. Download `quality-out/` from Google Drive, or open it in the Drive app.
2. Listen to `blind/L01_*.wav` and so on. Rank the versions for each line.
3. Only then open `blind/key.txt` to see which is which.
4. Tone comes from the reference clip. To change it, record a new `refs/<name>.wav` and `.txt`
   in the tone you want, then run the notebook again. It skips what is already made.

**Before saving this notebook back to GitHub:** Edit > **Clear all outputs**.